# Lab 6 — Amazon Aspect Sentiment: Mini-Kaggle con LLM Local

**Tarea:** dada una reseña de Amazon Electronics, clasificar el sentimiento para dos aspectos:
- **Calidad del producto** (`pos` / `neg`)
- **Entrega / envío** (`pos` / `neg`)

**Tu palanca:** el **prompt**. No hay entrenamiento. Gana quien consiga que el modelo responda de forma más precisa y consistente.

**Métrica:** F1 global = (F1 calidad + F1 entrega) / 2 · Leaderboard: URL del profesor

In [1]:
import re, time, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import requests
from pathlib import Path
from sklearn.metrics import f1_score, classification_report

SERVER_URL = 'http://localhost:8000'
OLLAMA_URL = 'http://localhost:11434'

# ── Datos ─────────────────────────────────────────────────────────────────────
try:
    import google.colab; _DATA = Path('/content/data')
except ImportError:
    _DATA = Path('../labs/lab6/data')
_DATA.mkdir(parents=True, exist_ok=True)

for _f in ('train.csv', 'public_test.csv'):
    if not (_DATA / _f).exists():
        (_DATA / _f).write_bytes(
            requests.get(f'{SERVER_URL}/data/{_f}', timeout=30).content)

train = pd.read_csv(_DATA / 'train.csv')
test  = pd.read_csv(_DATA / 'public_test.csv')

# ── LLM ───────────────────────────────────────────────────────────────────────
def call_llm(prompt):
    r = requests.post(
        f'{OLLAMA_URL}/api/generate',
        json={'model': MODEL, 'prompt': prompt,
              'stream': False, 'options': {'num_predict': 60, 'temperature': 0}},
        timeout=120)
    r.raise_for_status()
    return r.json()['response'].strip()

def parse_aspects(response):
    """Extrae (quality, delivery) de la respuesta del LLM."""
    text = response.lower()
    quality = delivery = 'neg'
    for line in text.split('\n'):
        line = line.strip()
        if line.startswith('quality'):  quality  = 'pos' if 'pos' in line else 'neg'
        if line.startswith('delivery'): delivery = 'pos' if 'pos' in line else 'neg'
    return quality, delivery

# ── Evaluación y envío ────────────────────────────────────────────────────────
def evaluar(prompt_fn, n=20):
    sample = train.sample(n=n, random_state=42)
    yq_true, yq_pred, yd_true, yd_pred = [], [], [], []
    for _, row in sample.iterrows():
        q, d = parse_aspects(call_llm(prompt_fn(row['text'])))
        yq_true.append(row['quality']);  yq_pred.append(q)
        yd_true.append(row['delivery']); yd_pred.append(d)
    f1q = f1_score(yq_true, yq_pred, pos_label='pos')
    f1d = f1_score(yd_true, yd_pred, pos_label='pos')
    print(f'F1 calidad : {f1q:.4f}')
    print(f'F1 entrega : {f1d:.4f}')
    print(f'F1 global  : {(f1q+f1d)/2:.4f}  (sobre {n} reseñas)')

def enviar(prompt_fn):
    payload, t0 = [], time.time()
    print(f'Clasificando {len(test)} reseñas...')
    for i, (_, row) in enumerate(test.iterrows()):
        q, d = parse_aspects(call_llm(prompt_fn(row['text'])))
        payload.append({'id': int(row['id']), 'quality': q, 'delivery': d})
        if (i + 1) % 10 == 0:
            eta = (time.time()-t0)/(i+1) * (len(test)-i-1)
            print(f'  {i+1}/{len(test)}  ETA {eta:.0f}s')
    r = requests.post(f'{SERVER_URL}/submit',
                      json={'team': ALUMNO, 'predictions': payload}, timeout=30)
    if r.status_code == 429:
        print(f'⏳ {r.json()["detail"]}')
    else:
        d = r.json()
        print(f"✅ F1 global: {d['f1']:.4f}  |  calidad: {d['f1_quality']:.4f}  "
              f"entrega: {d['f1_delivery']:.4f}  |  Rank: #{d['rank']}")

print(f'Train: {len(train):,}  |  Test: {len(test):,}')

Train: 1,500  |  Test: 88


---
## ✏️ Tu configuración

In [2]:
ALUMNO = 'pablo'        # <-- tu nombre o alias
MODEL  = 'llama3.2:3b' # <-- el profesor indicará qué modelo usar

# Comprueba que Ollama responde
print(call_llm('Di solo: OK'))

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 61] Connection refused"))

---
## 1. ¿Cómo son las reseñas?

El conjunto de entrenamiento tiene etiquetas visibles. Úsalas para entender el problema antes de escribir tu prompt.

In [ ]:
print(f'Longitud media: {train.text.str.split().str.len().mean():.0f} palabras\n')
for q, d, emoji in [('pos','pos','✅📦'), ('pos','neg','✅❌'), ('neg','pos','❌📦'), ('neg','neg','❌❌')]:
    row = train[(train.quality==q) & (train.delivery==d)].iloc[0]
    print(f'{emoji}  quality={q}  delivery={d}')
    print(f'   {row.text[:200]}...\n')

---
## 2. Baseline

Un prompt directo que le pide al modelo responder en formato fijo.

El parser busca las líneas `quality: pos/neg` y `delivery: pos/neg`. Si el modelo no sigue el formato exacto, asigna `neg` por defecto.

In [ ]:
def baseline_prompt(text):
    return f"""Classify this Amazon review's sentiment for two aspects.

Review: {text}

Reply ONLY with these two lines:
quality: pos
delivery: pos

Use pos/neg. If delivery is not mentioned, write neg."""

evaluar(baseline_prompt)

---
## 3. Tu prompt

Modifica la función para mejorar el resultado. Algunas ideas:

- **Define los aspectos** — ¿qué cuenta como "calidad"? ¿qué cuenta como "entrega"?
- **Few-shot** — incluye 1-2 ejemplos completos (reseña + respuesta correcta)
- **Chain-of-thought** — pide al modelo que razone antes de responder
- **Casos borde** — ¿qué debe responder si no se menciona la entrega?

Usa `evaluar` para probar localmente sobre 20 reseñas antes de enviar.

---
## Few-shot — ejemplos listos para usar

La variable `EXAMPLES` contiene un ejemplo por cada combinación posible, formateados para pegarlos directamente en tu prompt.
Incluir ejemplos reales suele ser el mayor salto de F1.

In [ ]:
# Ejemplos etiquetados — uno por combinación (quality × delivery)
# Cambia iloc[N] si quieres usar otro ejemplo del train
_combos = [('pos','pos'), ('pos','neg'), ('neg','pos'), ('neg','neg')]
_rows   = [train[(train.quality==q) & (train.delivery==d)].iloc[0]
           for q, d in _combos]

EXAMPLES = '\n\n'.join(
    f"Review: {r.text[:250]}\nquality: {q}\ndelivery: {d}"
    for r, (q, d) in zip(_rows, _combos)
)

print(EXAMPLES)

In [ ]:
def mi_prompt(text):
    return f"""Classify Amazon reviews. Here are examples:

{EXAMPLES}

Now classify this review:
Review: {' '.join(text.split()[:300])}

Reply ONLY with:
quality: pos
delivery: pos

Use pos for positive, neg for negative. If delivery is not mentioned, write neg."""

evaluar(mi_prompt)


In [ ]:
# Envía cuando estés listo — tarda varios minutos
enviar(mi_prompt)

---
## Tips

- **El formato es crítico** — el parser busca `quality:` y `delivery:` al inicio de línea; si el modelo añade texto extra antes, fallará
- **Few-shot funciona** — añadir 1-2 ejemplos en el prompt suele mejorar mucho la consistencia del formato
- **El leaderboard desglosa ambos aspectos** — si ves F1 calidad alto pero F1 entrega bajo, tu prompt está ignorando la entrega
- **20 reseñas locales tienen varianza** — si el F1 es inestable entre ejecuciones, el prompt es frágil; hazlo más explícito

¡Buena suerte! 🏆